# Alpamayo 1.5: Local Offline Inference with Axis Debug

This notebook runs local offline inference and adds coordinate-system diagnostics to compare predicted trajectories against GT under multiple axis transforms.

This notebook is intended to debug the mismatch between predicted trajectories and GT trajectories.

It provides:
- local offline inference
- visualization of all input images used for inference
- raw GT vs Pred plotting
- multiple GT coordinate transform comparisons
- diagnostic minADE under transformed GT
- best candidate transform selection

In [ ]:
import os
import sys
from pathlib import Path

repo_root = Path.cwd().parent
src_path = repo_root / 'src'
if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
from transformers import BitsAndBytesConfig

from alpamayo1_5 import helper
from alpamayo1_5.load_offline_dataset import load_offline_dataset
from alpamayo1_5.models.alpamayo1_5 import Alpamayo1_5

In [ ]:
# ===== Local paths =====
CLIP_DIR = Path('/workspace/dataset/')
MODEL_PATH = Path('/root/.cache/huggingface/hub/models--nvidia--Alpamayo-1.5-10B/snapshots/f11cd25b758ab560114019b555dde2a8b92d88b4')
PROCESSOR_PATH = Path('/root/.cache/huggingface/hub/models--Qwen--Qwen3-VL-8B-Instruct/snapshots/0c351dd01ed87e9c1b53cbc748cba10e6187ff3b')
COSMOS_PROCESSOR_PATH = Path('/root/.cache/huggingface/hub/models--nvidia--Cosmos-Reason2-8B/snapshots/f715d875a8a99a0a2b65aa28633afd9417e46bd9')

# ===== Inference config =====
DEVICE = 'cuda'
T0_SOD = 175490.23
NUM_HISTORY_STEPS = 16
NUM_FUTURE_STEPS = 64
TIME_STEP = 0.1
NUM_FRAMES = 4
FPS = 30.0
FRAME0_GPS_TIME_SOD = 175484.98
NUM_TRAJ_SAMPLES = 1
TEMPERATURE = 0.6
TOP_P = 0.98
MAX_GENERATION_LENGTH = 256

os.environ['ALPAMAYO_VLM_PROCESSOR_PATH'] = str(COSMOS_PROCESSOR_PATH)

print('CLIP_DIR =', CLIP_DIR)
print('MODEL_PATH =', MODEL_PATH)
print('PROCESSOR_PATH =', PROCESSOR_PATH)
print('ALPAMAYO_VLM_PROCESSOR_PATH =', os.environ['ALPAMAYO_VLM_PROCESSOR_PATH'])
print('DEVICE =', DEVICE)
print('T0_SOD =', T0_SOD)

### Load model and processor

In [ ]:
quantization_config = BitsAndBytesConfig(
    load_in_8bit=True,
    llm_int8_threshold=6.0,
    llm_int8_enable_fp32_cpu_offload=False,
)

model = Alpamayo1_5.from_pretrained(
    str(MODEL_PATH),
    dtype=torch.bfloat16,
    device_map='cuda:0',
    quantization_config=quantization_config,
)

processor = helper.get_processor(
    model.tokenizer,
    processor_name_or_path=PROCESSOR_PATH,
)

print('Model and processor loaded.')

### Load local offline data

In [ ]:
data = load_offline_dataset(
    clip_dir=CLIP_DIR,
    t0_sod=T0_SOD,
    num_history_steps=NUM_HISTORY_STEPS,
    num_future_steps=NUM_FUTURE_STEPS,
    time_step=TIME_STEP,
    num_frames=NUM_FRAMES,
    fps=FPS,
    frame0_gps_time_sod=FRAME0_GPS_TIME_SOD,
    debug=True,
)

print('Offline dataset loaded.')
print('camera_indices:', data['camera_indices'].tolist())
print('image_frames shape:', tuple(data['image_frames'].shape))
print('ego_history_xyz shape:', tuple(data['ego_history_xyz'].shape))
print('ego_future_xyz shape:', tuple(data['ego_future_xyz'].shape))
print('requested frame indices:', data['video_frame_indices'][0].tolist())
print('actual frame indices:', data['actual_video_frame_indices'][0].tolist())

### Show all sampled images used for inference

In [ ]:
image_frames = data['image_frames'].cpu().numpy()  # [N_cam, T, C, H, W]
camera_indices = data['camera_indices'].cpu().tolist()
requested_indices = data['video_frame_indices'].cpu().numpy()
actual_indices = data['actual_video_frame_indices'].cpu().numpy()

num_cams, num_frames, _, _, _ = image_frames.shape
camera_names = [helper.CAMERA_DISPLAY_NAMES.get(i, f'Camera {i}') for i in camera_indices]

fig, axes = plt.subplots(num_cams, num_frames, figsize=(4 * num_frames, 3.5 * num_cams))

if num_cams == 1 and num_frames == 1:
    axes = np.array([[axes]])
elif num_cams == 1:
    axes = np.array([axes])
elif num_frames == 1:
    axes = np.array([[ax] for ax in axes])

for cam_idx in range(num_cams):
    for t_idx in range(num_frames):
        ax = axes[cam_idx, t_idx]
        frame = np.transpose(image_frames[cam_idx, t_idx], (1, 2, 0))
        ax.imshow(frame)
        ax.axis('off')
        ax.set_title(
            f'{camera_names[cam_idx]}\n'
            f't={t_idx}, req={requested_indices[cam_idx, t_idx]}, actual={actual_indices[cam_idx, t_idx]}'
        )

plt.tight_layout()
plt.show()

### Prepare chat template

In [ ]:
messages = helper.create_message(
    frames=data['image_frames'].flatten(0, 1),
    camera_indices=data['camera_indices'],
)

inputs = processor.apply_chat_template(
    messages,
    tokenize=True,
    add_generation_prompt=False,
    continue_final_message=True,
    return_dict=True,
    return_tensors='pt',
)

print('seq length:', inputs.input_ids.shape)

model_inputs = {
    'tokenized_data': inputs,
    'ego_history_xyz': data['ego_history_xyz'],
    'ego_history_rot': data['ego_history_rot'],
}

model_inputs = helper.to_device(model_inputs, DEVICE)

### Run inference

In [ ]:
if DEVICE.startswith('cuda'):
    torch.cuda.manual_seed_all(42)
    autocast_context = torch.autocast('cuda', dtype=torch.bfloat16)
else:
    autocast_context = torch.autocast(device_type=DEVICE, enabled=False)

with autocast_context:
    pred_xyz, pred_rot, extra = model.sample_trajectories_from_data_with_vlm_rollout(
        data=model_inputs,
        top_p=TOP_P,
        temperature=TEMPERATURE,
        num_traj_samples=NUM_TRAJ_SAMPLES,
        max_generation_length=MAX_GENERATION_LENGTH,
        return_extra=True,
    )

print('Inference done.')

### Show CoT and raw metric

In [ ]:
cot_value = extra['cot'][0]
if hasattr(cot_value, 'tolist'):
    cot_value = cot_value.tolist()

print('Chain-of-Causation:')
print(cot_value)

gt_xyz = data['ego_future_xyz'].cpu().numpy()[0, 0, :, :]  # [T, 3]
gt_xy = gt_xyz[:, :2].T  # [2, T]

pred_xyz_np = pred_xyz.cpu().numpy()[0, 0, :, :, :]  # [K, T, 3]
pred_xy = pred_xyz_np[:, :, :2].transpose(0, 2, 1)  # [K, 2, T]

diff = np.linalg.norm(pred_xy - gt_xy[None, ...], axis=1).mean(-1)
raw_min_ade = float(diff.min())

print('Raw minADE:', raw_min_ade, 'meters')

### Raw GT vs Pred plot

In [ ]:
plt.figure(figsize=(6, 6))
plt.plot(gt_xyz[:, 0], gt_xyz[:, 1], 'k-o', linewidth=2, markersize=3, label='GT(raw)')
for sample_idx in range(pred_xyz_np.shape[0]):
    plt.plot(
        pred_xyz_np[sample_idx, :, 0],
        pred_xyz_np[sample_idx, :, 1],
        '-o',
        linewidth=2,
        markersize=3,
        alpha=0.8,
        label=f'Pred {sample_idx}',
    )
plt.scatter([0.0], [0.0], c='red', marker='x', s=100, label='t0 ego')
plt.xlabel('x')
plt.ylabel('y')
plt.title(f'Raw GT vs Pred @ t0_sod={T0_SOD}, raw minADE={raw_min_ade:.3f}m')
plt.legend()
plt.grid(True, alpha=0.3)
plt.gca().set_aspect('equal', adjustable='box')
plt.tight_layout()
plt.show()

### Coordinate transform diagnostics

In [ ]:
gt_xy_raw = gt_xyz[:, :2]                 # [T, 2]
pred_xy0 = pred_xyz_np[0, :, :2]          # [T, 2]

def tf_identity(xy):
    return xy

def tf_swap(xy):
    return xy[:, [1, 0]]

def tf_x_neg(xy):
    return np.stack([-xy[:, 0],  xy[:, 1]], axis=1)

def tf_y_neg(xy):
    return np.stack([ xy[:, 0], -xy[:, 1]], axis=1)

def tf_swap_x_neg(xy):
    return np.stack([-xy[:, 1], xy[:, 0]], axis=1)

def tf_swap_y_neg(xy):
    return np.stack([xy[:, 1], -xy[:, 0]], axis=1)

def tf_both_neg(xy):
    return np.stack([-xy[:, 0], -xy[:, 1]], axis=1)

candidates = {
    'gt(x, y)': tf_identity(gt_xy_raw),
    'gt(y, x)': tf_swap(gt_xy_raw),
    'gt(-x, y)': tf_x_neg(gt_xy_raw),
    'gt(x, -y)': tf_y_neg(gt_xy_raw),
    'gt(-y, x)': tf_swap_x_neg(gt_xy_raw),
    'gt(y, -x)': tf_swap_y_neg(gt_xy_raw),
    'gt(-x, -y)': tf_both_neg(gt_xy_raw),
}

def ade_2d(pred_xy, gt_xy):
    return float(np.linalg.norm(pred_xy - gt_xy, axis=1).mean())

rows = []
for name, gt_xy_candidate in candidates.items():
    rows.append({
        'transform': name,
        'ade_vs_pred0': ade_2d(pred_xy0, gt_xy_candidate),
        'gt_final_x': float(gt_xy_candidate[-1, 0]),
        'gt_final_y': float(gt_xy_candidate[-1, 1]),
        'gt_mean_abs_x': float(np.mean(np.abs(gt_xy_candidate[:, 0]))),
        'gt_mean_abs_y': float(np.mean(np.abs(gt_xy_candidate[:, 1]))),
    })

compare_df = pd.DataFrame(rows).sort_values('ade_vs_pred0', ascending=True).reset_index(drop=True)
display(compare_df)

best_name = compare_df.iloc[0]['transform']
best_gt_xy = candidates[best_name]

print('Best candidate transform:', best_name)

print('\n=== Raw GT axis stats ===')
print('GT first 5 xy:')
print(gt_xy_raw[:5])
print('GT last 5 xy:')
print(gt_xy_raw[-5:])
print('GT final x:', float(gt_xy_raw[-1, 0]))
print('GT final y:', float(gt_xy_raw[-1, 1]))
print('GT mean abs x:', float(np.mean(np.abs(gt_xy_raw[:, 0]))))
print('GT mean abs y:', float(np.mean(np.abs(gt_xy_raw[:, 1]))))

print('\n=== Pred[0] axis stats ===')
print('Pred[0] first 5 xy:')
print(pred_xy0[:5])
print('Pred[0] last 5 xy:')
print(pred_xy0[-5:])
print('Pred[0] final x:', float(pred_xy0[-1, 0]))
print('Pred[0] final y:', float(pred_xy0[-1, 1]))
print('Pred[0] mean abs x:', float(np.mean(np.abs(pred_xy0[:, 0]))))
print('Pred[0] mean abs y:', float(np.mean(np.abs(pred_xy0[:, 1]))))

### Plot all GT transform candidates against Pred[0]

In [ ]:
plot_names = list(candidates.keys())
n = len(plot_names)
ncols = 3
nrows = int(np.ceil(n / ncols))

fig, axes = plt.subplots(nrows, ncols, figsize=(5 * ncols, 5 * nrows))
axes = np.array(axes).reshape(-1)

for ax, name in zip(axes, plot_names):
    gt_xy_candidate = candidates[name]
    candidate_ade = ade_2d(pred_xy0, gt_xy_candidate)

    ax.plot(pred_xy0[:, 0], pred_xy0[:, 1], 'r-o', label='Pred[0]', linewidth=2, markersize=3)
    ax.plot(gt_xy_candidate[:, 0], gt_xy_candidate[:, 1], 'k-o', label=name, linewidth=2, markersize=3)
    ax.scatter([0], [0], c='blue', marker='x', s=80)
    ax.set_title(f'{name}\nADE={candidate_ade:.3f}')
    ax.set_aspect('equal', adjustable='box')
    ax.grid(True, alpha=0.3)
    ax.legend()

for ax in axes[len(plot_names):]:
    ax.axis('off')

plt.tight_layout()
plt.show()

### Plot best GT transform against Pred[0]

In [ ]:
plt.figure(figsize=(6, 6))
plt.plot(pred_xy0[:, 0], pred_xy0[:, 1], 'r-o', linewidth=2, markersize=3, label='Pred[0]')
plt.plot(best_gt_xy[:, 0], best_gt_xy[:, 1], 'k-o', linewidth=2, markersize=3, label=best_name)
plt.scatter([0], [0], c='blue', marker='x', s=100, label='Origin')
plt.title(f'Best GT transform vs Pred[0]: {best_name}')
plt.xlabel('x')
plt.ylabel('y')
plt.grid(True, alpha=0.3)
plt.legend()
plt.gca().set_aspect('equal', adjustable='box')
plt.tight_layout()
plt.show()

### Diagnostic metric using GT -> (-y, x)

In [ ]:
gt_xy_diag = np.stack([-gt_xyz[:, 1], gt_xyz[:, 0]], axis=1)   # [T, 2]
pred_xy_diag = pred_xyz_np[:, :, :2]                           # [K, T, 2]

diff_diag = np.linalg.norm(pred_xy_diag - gt_xy_diag[None, :, :], axis=2).mean(axis=1)
diag_min_ade = float(diff_diag.min())

print('Raw minADE:', raw_min_ade)
print('Diagnostic minADE with GT -> (-y, x):', diag_min_ade)

### Diagnostic plot with GT -> (-y, x)

In [ ]:
plt.figure(figsize=(6, 6))
plt.plot(pred_xyz_np[0, :, 0], pred_xyz_np[0, :, 1], 'r-o', linewidth=2, markersize=3, label='Pred[0]')
plt.plot(gt_xy_diag[:, 0], gt_xy_diag[:, 1], 'k-o', linewidth=2, markersize=3, label='GT -> (-y, x)')
plt.scatter([0], [0], c='blue', marker='x', s=100, label='Origin')
plt.xlabel('x')
plt.ylabel('y')
plt.title(f'Diagnostic plot: GT -> (-y, x), diag minADE={diag_min_ade:.3f}m')
plt.grid(True, alpha=0.3)
plt.legend()
plt.gca().set_aspect('equal', adjustable='box')
plt.tight_layout()
plt.show()

### Quick result summary

In [ ]:
summary_df = pd.DataFrame([
    {
        't0_sod': T0_SOD,
        'raw_min_ade': raw_min_ade,
        'best_transform': best_name,
        'diag_min_ade_gt_to_minus_y_x': diag_min_ade,
        'camera_indices': ';'.join(map(str, data['camera_indices'].cpu().tolist())),
        'requested_frame_indices': ';'.join(map(str, data['video_frame_indices'][0].cpu().tolist())),
        'actual_frame_indices': ';'.join(map(str, data['actual_video_frame_indices'][0].cpu().tolist())),
        'cot': cot_value,
    }
])

display(summary_df)